In [1]:
!git clone https://github.com/Seloga26/Concurrente.git repo && cd repo


Cloning into 'repo'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 105 (delta 38), reused 102 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 415.48 KiB | 12.59 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [15]:
%cd /content/repo

/content/repo


In [16]:
!pip -q install pycuda

In [3]:
!bash repo/CUDA/colab_validate.sh

[colab] generando dataset...
Generado en data:
  candidates_W.npy
  generation_metadata.json
  labels.npy
  matrix_A.npy
  profile_F.npy
  profile_S.npy
  profile_T.npy
[colab] compilando scoring_cuda (nvcc)...
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
== mode=reference ==
  cuda       : best_k=0 auc_units=50 theta=0.497935940601  dev=Tesla T4  -> OK  (exacto vs python)
  cuda_pycuda: best_k=0 auc_units=50 theta=0.497935940601  dev=Tesla T4  -> OK  (exacto vs python)
  t_core(kernel): python=1.3292s  cuda=0.16004s  cuda_pycuda=0.00329s
== mode=benchmark ==
  cuda       : best_k=0 auc_units=50 theta=0.497935935855  dev=Tesla T4  -> OK  (exacto vs python)
  cuda_pycuda: best_k=0 auc_units=50 theta=0.497935920954  dev=Tesla T4  -> OK  (exacto vs python)
  t_core(kernel): python=1.6060s  cuda=0.00072s  cuda_pycuda=0.00093s
VALIDACION CUDA OK


In [17]:
!make -C CUDA

make: Entering directory '/content/repo/CUDA'
make: Nothing to be done for 'all'.
make: Leaving directory '/content/repo/CUDA'


In [18]:
!./CUDA/scoring_cuda \
  --N 50 \
  --K 100000 \
  --mode reference \
  --accum float64 \
  --algorithm literal \
  --tie-atol 1e-9 \
  --tie-rtol 1e-9 \
  --matrix-a data/matrix_A.npy \
  --profile-t data/profile_T.npy \
  --profile-s data/profile_S.npy \
  --profile-f data/profile_F.npy \
  --labels data/labels.npy \
  --candidates data/candidates_W.npy \
  --theta-policy class_mean_midpoint \
  --consistency-threshold 0.8

{"implementation": "cuda", "mode": "reference", "algorithm": "literal", "kernel_variant": "fused", "accum_dtype": "float64", "n_items": 50, "n_candidates": 100000, "best_k": 0, "best_w": [0.61811304092407227, 0.12117547541856766, 0.26071149110794067], "auc_units": 50, "auc_denominator": 50, "auc": 1, "scores": [0.35438312890155982, 0.31292717367132128, 0.29088730580301708, 0.38572634558332175, 0.34014779655940797, 0.66003804000235133, 0.64705856068575829, 0.66266613499518645, 0.65342388536547302, 0.6721010344385957], "theta": 0.4979359406005992, "consistency": 1, "consistency_threshold": 0.80000000000000004, "consistency_pass": true, "tie_atol": 1.0000000000000001e-09, "tie_rtol": 1.0000000000000001e-09, "theta_policy": "class_mean_midpoint", "t_core_seconds": 0.0042423682212829588, "t_search_seconds": 0.0046553602218627928, "cuda_block_size": 256, "cuda_grid_size": 391, "device": "Tesla T4"}


In [19]:
!python CUDA/scoring_pycuda.py \
  --N 50 \
  --K 100000 \
  --mode reference \
  --accum float64 \
  --algorithm literal \
  --tie-atol 1e-9 \
  --tie-rtol 1e-9 \
  --matrix-a data/matrix_A.npy \
  --profile-t data/profile_T.npy \
  --profile-s data/profile_S.npy \
  --profile-f data/profile_F.npy \
  --labels data/labels.npy \
  --candidates data/candidates_W.npy \
  --theta-policy class_mean_midpoint \
  --consistency-threshold 0.8 \
  --block-size 256

{"implementation": "cuda_pycuda", "mode": "reference", "algorithm": "literal", "kernel_variant": "fused", "accum_dtype": "float64", "n_items": 50, "n_candidates": 100000, "best_k": 0, "best_w": [0.6181130409240723, 0.12117547541856766, 0.2607114911079407], "auc_units": 50, "auc_denominator": 50, "auc": 1.0, "scores": [0.35438312890156, 0.3129271736713213, 0.290887305803017, 0.3857263455833219, 0.3401477965594081, 0.6600380400023512, 0.6470585606857582, 0.6626661349951863, 0.653423885365473, 0.6721010344385957], "theta": 0.4979359406005993, "consistency": 1.0, "consistency_threshold": 0.8, "consistency_pass": true, "tie_atol": 1e-09, "tie_rtol": 1e-09, "theta_policy": "class_mean_midpoint", "t_core_seconds": 0.005193696022033691, "t_search_seconds": 0.0055968317985534664, "cuda_block_size": 256, "cuda_grid_size": 391, "device": "Tesla T4"}


In [20]:
!ls CUDA
!ls data
!nvidia-smi

colab_validate.sh  npyio.o	    scoring_cuda	scoring.o
CUDA.ipynb	   README_colab.md  scoring_device.cuh	scoring_pycuda.py
Makefile	   runner.o	    scoring_kernel.cu
candidates_W.npy  generation_metadata.json  matrix_A.npy   profile_T.npy
genconfig.py	  __init__.py		    profile_F.npy  __pycache__
generate_data.py  labels.npy		    profile_S.npy
Tue Jun 16 22:50:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+===

In [21]:
!bash CUDA/colab_validate.sh

[colab] compilando scoring_cuda (nvcc)...
== mode=reference ==
  cuda       : best_k=0 auc_units=50 theta=0.497935940601  dev=Tesla T4  -> OK  (exacto vs python)
  cuda_pycuda: best_k=0 auc_units=50 theta=0.497935940601  dev=Tesla T4  -> OK  (exacto vs python)
  t_core(kernel): python=1.4396s  cuda=0.00308s  cuda_pycuda=0.00333s
== mode=benchmark ==
  cuda       : best_k=0 auc_units=50 theta=0.497935935855  dev=Tesla T4  -> OK  (exacto vs python)
  cuda_pycuda: best_k=0 auc_units=50 theta=0.497935920954  dev=Tesla T4  -> OK  (exacto vs python)
  t_core(kernel): python=2.4052s  cuda=0.00090s  cuda_pycuda=0.00084s
VALIDACION CUDA OK
